In [1]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
 
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

In [6]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

### COX assumption in Train data

In [7]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic    p  -log2(p)
MTV                       0.00 0.95      0.08
SUVpeak                   0.29 0.59      0.76
TLG                       0.22 0.64      0.65
age                       1.60 0.21      2.28
cavum_oris                0.00 1.00      0.01
charlson                  0.07 0.79      0.34
female                    0.17 0.68      0.56
histgrade_high            1.26 0.26      1.93
hpv_related               4.31 0.04      4.72
hypopharynx               0.00 0.99      0.01
larynx                    0.00 0.98      0.02
oropharynx                0.00 0.99      0.02
pack_years                0.71 0.40      1.32
uicc8_III-IV              0.45 0.50      0.99


In [8]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


In [9]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic    p  -log2(p)
MTV                       0.02 0.88      0.18
SUVpeak                   0.56 0.46      1.13
TLG                       0.06 0.81      0.30
age                       1.58 0.21      2.26
cavum_oris                0.08 0.77      0.37
charlson                  0.03 0.86      0.22
female                    0.35 0.56      0.85
histgrade_high            0.95 0.33      1.60
hpv_related               3.02 0.08      3.60
hypopharynx               0.10 0.75      0.41
larynx                    1.46 0.23      2.14
oropharynx                0.96 0.33      1.61
pack_years                0.39 0.53      0.92
uicc8_III-IV              0.08 0.78      0.36


In [10]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index([], dtype='object')


###### penalizer values essentially result in same result 

## Test dataset: MAASTRO 

In [11]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [12]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [13]:
# need to choose patient_id from MAASTRO_D1 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [14]:
# Merge MAASTRO_D1 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event
0,1,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342,8.83,1.0
3,4,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979,19.73,1.0
4,6,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782,13.27,1.0
95,111,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492,58.93,0.0


In [15]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [16]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event


In [17]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [18]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [19]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 14)
y_train:  (139,)


In [20]:
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [21]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 16)

In [22]:
# VIF dataframe 
vif_data = pd.DataFrame() 
vif_data["feature"] = X.columns 
  
# calculating VIF for each feature 
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                          for i in range(len(X.columns))] 

vif_data

,feature,VIF
0,age,1.136852
1,female,1.197250
2,cavum_oris,7.993011
3,oropharynx,60.199385
4,hypopharynx,11.118710
5,larynx,14.183474
6,histgrade_high,1.150479
7,hpv_related,4.309385
8,charlson,1.302350
9,pack_years,1.647441


# Standardization

In [23]:
original_X = X.copy()

In [24]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

## Standardize the data but not the categorical columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

### Separate the categorical and non-categorical columns
X_categorical = X[categorical_columns]
X_numeric = X.drop(categorical_columns, axis=1)
X_numeric_columns = X_numeric.columns
X_numeric_index = X_numeric.index

### Standardize non-categorical and then concat with the categorical
scaler = MinMaxScaler()  
X_numeric_std = scaler.fit_transform(X_numeric)
X_numeric_std = pd.DataFrame(X_numeric_std, columns=X_numeric_columns, index=X_numeric_index)
X_std = pd.concat([X_categorical, X_numeric_std], axis=1)

## Sort the order of the columns as it was in the clinical train 
X_std = X_std[original_X.columns]

In [25]:
# Standardize X_MAASTRO 
MAASTRO_new = X_MAASTRO.copy()
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [26]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = MAASTRO_new_std 

In [27]:
X_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,54.238356,1,0,1,0,0,1,0.0,0,0.000000,0.0,14.473272,7.934,86.228420
1,54.539726,0,0,0,0,1,0,0.0,1,27.404795,0.0,5.044678,1.656,7.040100
2,59.019178,0,1,0,0,0,1,0.0,1,41.019178,1.0,7.839043,14.502,83.569669
3,70.726027,0,0,0,0,1,0,0.0,1,37.500000,0.0,2.880631,2.440,5.567091
4,67.865753,0,0,0,0,1,0,0.0,1,53.000000,0.0,5.402006,3.668,16.150550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,60.435616,0,0,1,0,0,1,1.0,0,0.000000,0.0,9.290139,3.650,26.280140
135,68.794521,0,0,1,0,0,1,1.0,0,0.000000,1.0,7.172883,18.967,101.754834
136,57.498630,0,0,1,0,0,1,1.0,1,39.498630,0.0,13.873187,6.370,66.273201
137,65.684932,0,0,1,0,0,1,1.0,1,71.527397,1.0,7.507419,12.443,71.832443


In [28]:
X_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,0.365347,1,0,1,0,0,1,0.0,0,0.000000,0.0,0.435155,0.074159,0.044798
1,0.373041,0,0,0,0,1,0,0.0,1,0.213981,0.0,0.089331,0.008512,0.001900
2,0.487409,0,1,0,0,0,1,0.0,1,0.320284,1.0,0.191823,0.142839,0.043358
3,0.786304,0,0,0,0,1,0,0.0,1,0.292806,0.0,0.009957,0.016710,0.001102
4,0.713276,0,0,0,0,1,0,0.0,1,0.413832,0.0,0.102437,0.029551,0.006835
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,0.523573,0,0,1,0,0,1,1.0,0,0.000000,0.0,0.245047,0.029363,0.012323
135,0.736989,0,0,1,0,0,1,1.0,0,0.000000,1.0,0.167389,0.189529,0.053209
136,0.448587,0,0,1,0,0,1,1.0,1,0.308411,0.0,0.413145,0.057805,0.033988
137,0.657597,0,0,1,0,0,1,1.0,1,0.558497,1.0,0.179660,0.121309,0.037000


In [29]:
MAASTRO_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623
1,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700
2,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342
3,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979
4,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782
95,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868
96,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274
97,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492


In [30]:
MAASTRO_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,0.384793,0,0,1,0,0,1,1,1,0.000000,0,0.470560,0.230038,0.140892
1,0.384793,0,0,1,0,0,0,0,0,0.156163,1,0.228146,0.050381,0.018119
2,0.384793,0,0,1,0,0,0,0,1,0.046849,1,0.398581,0.072664,0.038519
3,0.537983,1,0,0,0,1,1,0,1,0.351367,1,0.220934,0.073887,0.023434
4,0.767767,0,0,1,0,0,1,1,1,0.460681,0,0.263159,0.150525,0.056396
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,0.665641,1,0,0,0,1,0,0,0,0.429449,1,1.053738,0.055086,0.076361
95,0.589046,0,0,0,0,1,0,0,1,1.358619,1,0.382644,0.066296,0.035582
96,0.589046,0,0,1,0,0,1,1,1,0.000000,1,0.232370,0.163554,0.053664
97,0.359261,0,0,1,0,0,1,1,0,0.000000,0,0.424553,0.095564,0.054008


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [141]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min-max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-21 16:14:53,526] A new study created in memory with name: no-name-b63464ab-cda0-4ae6-8e3e-00d88dad8d21


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-21 16:14:53,711] A new study created in memory with name: no-name-c3ff4653-f8d3-4dd7-a6fc-eb9ce2f996f3


Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.630901287553648
[I 2024-04-21 16:14:53,707] Trial 0 finished with value: 0.6371137109369622 and parameters: {}. Best is trial 0 with value: 0.6371137109369622.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6371137109369622], datetime_start=datetime.datetime(2024, 4, 21, 16, 14, 53, 550947), datetime_complete=datetime.datetime(2024, 4, 21, 16, 14, 53, 707231), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6371137109369622


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.5623041979949875
Fold 2 IBS: 0.2636542638834043
Fold 3 IBS: 0.18152300182294628
Fold 4 IBS: 0.2906666179111608
Fold 5 IBS: 0.21154504298347718
[I 2024-04-21 16:14:53,922] Trial 0 finished with value: 0.3019386249191952 and parameters: {}. Best is trial 0 with value: 0.3019386249191952.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.3019386249191952], datetime_start=datetime.datetime(2024, 4, 21, 16, 14, 53, 736569), datetime_complete=datetime.datetime(2024, 4, 21, 16, 14, 53, 922084), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.3019386249191952


In [142]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [143]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.637
train_ibs:  0.302


#### Test

In [144]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [145]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.563
IBS score: 0.277


In [110]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [37]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [38]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-21 15:25:11,352] A new study created in memory with name: no-name-e74760c2-91ad-4d90-91df-a360eeaaf53d


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-21 15:25:11,589] A new study created in memory with name: no-name-ce04d780-b487-474d-8f6d-1dc31b26c0b2


Fold 1 C-index: 0.5858453473132372
Fold 2 C-index: 0.5959349593495935
Fold 3 C-index: 0.6361731843575419
[I 2024-04-21 15:25:11,580] Trial 0 finished with value: 0.6059844970067908 and parameters: {}. Best is trial 0 with value: 0.6059844970067908.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6059844970067908], datetime_start=datetime.datetime(2024, 4, 21, 15, 25, 11, 422127), datetime_complete=datetime.datetime(2024, 4, 21, 15, 25, 11, 580543), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6059844970067908


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24820216468962097
Fold 2 IBS: 0.23351868202031184
Fold 3 IBS: 0.2346623445671507
[I 2024-04-21 15:25:11,973] Trial 0 finished with value: 0.23879439709236117 and parameters: {}. Best is trial 0 with value: 0.23879439709236117.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23879439709236117], datetime_start=datetime.datetime(2024, 4, 21, 15, 25, 11, 645357), datetime_complete=datetime.datetime(2024, 4, 21, 15, 25, 11, 972844), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23879439709236117


In [39]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [40]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.606
train_ibs:  0.239


#### Test

In [41]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [42]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.657


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.224


In [43]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [44]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-21 15:25:12,387] A new study created in memory with name: no-name-f3005775-a7f2-4eb1-95ed-0cfdbfb84c3c


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.580602883355177
Fold 2 C-index: 0.6634146341463415
Fold 3 C-index: 0.5391061452513967
[I 2024-04-21 15:25:12,794] Trial 0 finished with value: 0.5943745542509716 and parameters: {}. Best is trial 0 with value: 0.5943745542509716.


[I 2024-04-21 15:25:12,809] A new study created in memory with name: no-name-5843031f-2827-4335-85be-9a4d0b6ca9a9




* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5943745542509716], datetime_start=datetime.datetime(2024, 4, 21, 15, 25, 12, 420904), datetime_complete=datetime.datetime(2024, 4, 21, 15, 25, 12, 794511), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5943745542509716


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2475305706612343
Fold 2 IBS: 0.27754301055956615
Fold 3 IBS: 0.2912429350332853
[I 2024-04-21 15:25:13,580] Trial 0 finished with value: 0.2721055054180286 and parameters: {}. Best is trial 0 with value: 0.2721055054180286.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2721055054180286], datetime_start=datetime.datetime(2024, 4, 21, 15, 25, 12, 863581), datetime_complete=datetime.datetime(2024, 4, 21, 15, 25, 13, 580162), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2721055054180286


In [45]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [46]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.594
train_ibs:  0.272


#### Test

In [47]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [48]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.562


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.275


In [49]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [50]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-21 15:25:14,202] A new study created in memory with name: no-name-c2f73ef5-90fb-47ea-b30f-878fa351aa83


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.581913499344692
Fold 2 C-index: 0.6634146341463415
Fold 3 C-index: 0.5391061452513967
[I 2024-04-21 15:25:14,941] Trial 0 finished with value: 0.5948114262474767 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.5948114262474767.
Fold 1 C-index: 0.583224115334207
Fold 2 C-index: 0.6634146341463415
Fold 3 C-index: 0.5335195530726257
[I 2024-04-21 15:25:15,437] Trial 1 finished with value: 0.593386100851058 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.5948114262474767.
Fold 1 C-index: 0.583224115334207
Fold 2 C-index: 0.6634146341463415
Fold 3 C-index: 0.5335195530726257
[I 2024-04-21 15:25:15,841] Trial 2 finished with value: 0.593386100851058 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.5948114262474767.
Fold 1 C-index: 0.583224115334207
Fold 2 C-index: 0.6634146341463415
Fold 3 C-index: 0.5349162011173184
[I 2024-04-21 15:25:16,169] Trial 3 finished with value: 0.593

Fold 1 C-index: 0.583224115334207
Fold 2 C-index: 0.6634146341463415
Fold 3 C-index: 0.5335195530726257
[I 2024-04-21 15:25:23,724] Trial 30 finished with value: 0.593386100851058 and parameters: {'l1_ratio': 0.32170963481394405}. Best is trial 14 with value: 0.6059844970067908.
Fold 1 C-index: 0.5858453473132372
Fold 2 C-index: 0.5959349593495935
Fold 3 C-index: 0.6361731843575419
[I 2024-04-21 15:25:23,871] Trial 31 finished with value: 0.6059844970067908 and parameters: {'l1_ratio': 0.00017291237040205354}. Best is trial 14 with value: 0.6059844970067908.
Fold 1 C-index: 0.5858453473132372
Fold 2 C-index: 0.5959349593495935
Fold 3 C-index: 0.6361731843575419
[I 2024-04-21 15:25:24,220] Trial 32 finished with value: 0.6059844970067908 and parameters: {'l1_ratio': 0.04949751601433792}. Best is trial 14 with value: 0.6059844970067908.
Fold 1 C-index: 0.581913499344692
Fold 2 C-index: 0.5959349593495935
Fold 3 C-index: 0.6361731843575419
[I 2024-04-21 15:25:24,507] Trial 33 finished wit

Fold 1 C-index: 0.5845347313237221
Fold 2 C-index: 0.6634146341463415
Fold 3 C-index: 0.5335195530726257
[I 2024-04-21 15:25:31,976] Trial 60 finished with value: 0.5938229728475631 and parameters: {'l1_ratio': 0.1853725643519477}. Best is trial 14 with value: 0.6059844970067908.
Fold 1 C-index: 0.5858453473132372
Fold 2 C-index: 0.5959349593495935
Fold 3 C-index: 0.6361731843575419
[I 2024-04-21 15:25:32,064] Trial 61 finished with value: 0.6059844970067908 and parameters: {'l1_ratio': 0.08688025702675667}. Best is trial 14 with value: 0.6059844970067908.
Fold 1 C-index: 0.5858453473132372
Fold 2 C-index: 0.5959349593495935
Fold 3 C-index: 0.6361731843575419
[I 2024-04-21 15:25:32,180] Trial 62 finished with value: 0.6059844970067908 and parameters: {'l1_ratio': 0.09349654471318652}. Best is trial 14 with value: 0.6059844970067908.
Fold 1 C-index: 0.5858453473132372
Fold 2 C-index: 0.5959349593495935
Fold 3 C-index: 0.6361731843575419
[I 2024-04-21 15:25:32,273] Trial 63 finished with

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.2475783479188048
Fold 2 IBS: 0.2775672556954279
Fold 3 IBS: 0.2911897513240198
[I 2024-04-21 15:25:41,260] Trial 0 finished with value: 0.2721117849794175 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.2721117849794175.
Fold 1 IBS: 0.24760277316282148
Fold 2 IBS: 0.2775696953239809
Fold 3 IBS: 0.29138596286076807
[I 2024-04-21 15:25:41,784] Trial 1 finished with value: 0.2721861437825235 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.2721117849794175.
Fold 1 IBS: 0.24759599016705242
Fold 2 IBS: 0.27757543143620833
Fold 3 IBS: 0.291488754490802
[I 2024-04-21 15:25:42,252] Trial 2 finished with value: 0.27222005869802096 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.2721117849794175.
Fold 1 IBS: 0.247511526612108
Fold 2 IBS: 0.277573566272216
Fold 3 IBS: 0.2913565288303236
[I 2024-04-21 15:25:42,889] Trial 3 finished with value: 0.2721472072382159 and parameters: {'l1_ratio

Fold 1 IBS: 0.24607115587517053
Fold 2 IBS: 0.23348684240516673
Fold 3 IBS: 0.23317323279498642
[I 2024-04-21 15:25:53,085] Trial 32 finished with value: 0.2375770770251079 and parameters: {'l1_ratio': 0.06207691234646743}. Best is trial 14 with value: 0.23697501317890082.
Fold 1 IBS: 0.24738945648035607
Fold 2 IBS: 0.2775781942344326
Fold 3 IBS: 0.2915640914550656
[I 2024-04-21 15:25:53,488] Trial 33 finished with value: 0.27217724738995147 and parameters: {'l1_ratio': 0.1778000985129503}. Best is trial 14 with value: 0.23697501317890082.
Fold 1 IBS: 0.24758657901222147
Fold 2 IBS: 0.2775716626924408
Fold 3 IBS: 0.29144638579336674
[I 2024-04-21 15:25:54,103] Trial 34 finished with value: 0.272201542499343 and parameters: {'l1_ratio': 0.2432080643136032}. Best is trial 14 with value: 0.23697501317890082.
Fold 1 IBS: 0.2456010316322291
Fold 2 IBS: 0.23351172313511426
Fold 3 IBS: 0.2328392623441898
[I 2024-04-21 15:25:54,287] Trial 35 finished with value: 0.23731733903717775 and paramet

Fold 1 IBS: 0.2473183008519733
Fold 2 IBS: 0.27759486992242516
Fold 3 IBS: 0.291502575182032
[I 2024-04-21 15:26:03,406] Trial 63 finished with value: 0.27213858198547686 and parameters: {'l1_ratio': 0.16894030439294788}. Best is trial 14 with value: 0.23697501317890082.
Fold 1 IBS: 0.24493635179953974
Fold 2 IBS: 0.23356967109590185
Fold 3 IBS: 0.2323655010409278
[I 2024-04-21 15:26:03,550] Trial 64 finished with value: 0.2369571746454565 and parameters: {'l1_ratio': 0.10284788949385718}. Best is trial 64 with value: 0.2369571746454565.
Fold 1 IBS: 0.24696479523533535
Fold 2 IBS: 0.23347315466846255
Fold 3 IBS: 0.23380345340130332
[I 2024-04-21 15:26:03,740] Trial 65 finished with value: 0.23808046776836708 and parameters: {'l1_ratio': 0.0340532859434732}. Best is trial 64 with value: 0.2369571746454565.
Fold 1 IBS: 0.246526863328302
Fold 2 IBS: 0.23357681176987877
Fold 3 IBS: 0.2323193467368943
[I 2024-04-21 15:26:04,128] Trial 66 finished with value: 0.2374743406116917 and parameter

Fold 1 IBS: 0.24751890505038296
Fold 2 IBS: 0.277581505409534
Fold 3 IBS: 0.2915607477694774
[I 2024-04-21 15:26:13,400] Trial 95 finished with value: 0.27222038607646476 and parameters: {'l1_ratio': 0.19634020969378}. Best is trial 81 with value: 0.23694266226705987.
Fold 1 IBS: 0.24673406364790046
Fold 2 IBS: 0.2336140106263108
Fold 3 IBS: 0.23210292905826285
[I 2024-04-21 15:26:13,664] Trial 96 finished with value: 0.23748366777749133 and parameters: {'l1_ratio': 0.11755546648562562}. Best is trial 81 with value: 0.23694266226705987.
Fold 1 IBS: 0.2473348142267078
Fold 2 IBS: 0.2775518986735307
Fold 3 IBS: 0.2915166637703804
[I 2024-04-21 15:26:14,113] Trial 97 finished with value: 0.2721344588902063 and parameters: {'l1_ratio': 0.17092443812719485}. Best is trial 81 with value: 0.23694266226705987.
Fold 1 IBS: 0.24756210841779558
Fold 2 IBS: 0.2775473570094509
Fold 3 IBS: 0.2914778967125745
[I 2024-04-21 15:26:14,701] Trial 98 finished with value: 0.2721957873799403 and parameters:

In [51]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [52]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.606
train_ibs:  0.237


#### Test

In [53]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [54]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.1015556329868651)

test_cindex : 0.657


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.10390511312334547)

test_ibs:  0.219


In [55]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [56]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-21 15:26:15,269] A new study created in memory with name: no-name-52926fcb-c3c9-4067-a49b-bf5a9cfd7abb


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6094364351245085
Fold 2 C-index: 0.5349593495934959
Fold 3 C-index: 0.5495810055865922
[I 2024-04-21 15:26:18,179] Trial 0 finished with value: 0.5646589301015322 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.5646589301015322.
Fold 1 C-index: 0.6081258191349934
Fold 2 C-index: 0.5414634146341464
Fold 3 C-index: 0.5614525139664804
[I 2024-04-21 15:26:19,924] Trial 1 finished with value: 0.5703472492452067 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.11413161543947781, 'warm_start': False}. Best is trial 1 with value: 0.570347249245206

Fold 2 C-index: 0.7707317073170732
Fold 3 C-index: 0.6815642458100558
[I 2024-04-21 15:26:55,634] Trial 17 finished with value: 0.6889916174032326 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 2, 'min_samples_leaf': 1, 'max_depth': 2, 'n_estimators': 493, 'oob_score': True, 'max_samples': 0.4399216700525951, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.09373644341192987, 'warm_start': True}. Best is trial 14 with value: 0.6982643510327705.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
[I 2024-04-21 15:26:58,461] Trial 18 finished with value: 0.5 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 7, 'min_samples_leaf': 7, 'max_depth': 9, 'n_estimators': 455, 'oob_score': True, 'max_samples': 0.20703029814340784, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.38335524316027697, 'warm_start': True}. Best is trial 14 with value: 0.6982643510327705.
Fold 1 C-index: 0.6041939711664482
Fold 2 C-index: 0.8373983739837398
Fold 3 C-index: 0.7793

Fold 1 C-index: 0.6225425950196593
Fold 2 C-index: 0.7398373983739838
Fold 3 C-index: 0.6857541899441341
[I 2024-04-21 15:27:30,509] Trial 34 finished with value: 0.6827113944459257 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 4, 'min_samples_leaf': 2, 'max_depth': 5, 'n_estimators': 378, 'oob_score': True, 'max_samples': 0.3921322757689111, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.11898813720646406, 'warm_start': True}. Best is trial 19 with value: 0.7403073180295786.
Fold 1 C-index: 0.6094364351245085
Fold 2 C-index: 0.7967479674796748
Fold 3 C-index: 0.7402234636871509
[I 2024-04-21 15:27:33,136] Trial 35 finished with value: 0.715469288763778 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 7, 'min_samples_leaf': 7, 'max_depth': 9, 'n_estimators': 419, 'oob_score': True, 'max_samples': 0.6846754529760358, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.03214075386929217, 'warm_start': True}. Best is trial 19 with value: 0.7403073180295786

KeyboardInterrupt: 

In [ ]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [ ]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

#### Test

In [ ]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

In [ ]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

In [ ]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [ ]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [ ]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


In [ ]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [ ]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

#### Test

In [ ]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [ ]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

In [ ]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [ ]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

In [ ]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [ ]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

#### Test

In [ ]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [ ]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

In [ ]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

### Example

In [61]:
# Min Max Standardization
excluded_columns = ['female', 
                    'cavum_oris',
                    'oropharynx',
                    'hypopharynx',
                    'larynx',
                    'histgrade_high',
                    'hpv_related',
                    'charlson',
                    'uicc8_III-IV'
                   ]
excluded_columns = set(excluded_columns).intersection(X.columns)

scaler = MinMaxScaler() 
X_train_included = X.drop(excluded_columns, axis=1)
X_test_included = X.drop(excluded_columns, axis=1)

In [63]:
X_train_included_std = scaler.fit_transform(X_train_included)
X_test_included_std = scaler.transform(X_test_included)

# Concatenation
X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
X_train_std = pd.concat([X_train_std_df, X[excluded_columns]], axis=1)

X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
X_test_std = pd.concat([X_test_std_df, X[excluded_columns]], axis=1)

In [84]:
X_std = X_train_std[X.columns]
MAASTRO_std = MAASTRO_new_std[X.columns]

In [85]:
model_with_standardization = ComponentwiseGradientBoostingSurvivalAnalysis().fit(X_std, y)

In [106]:
model_with_standardization.score(X_std, y)

0.6724952741020794

In [86]:
model_with_standardization.score(MAASTRO_std, y_MAASTRO)

0.6006072315760419

In [94]:
lower, upper = np.percentile(y["time"], [10, 90])    
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in model_with_standardization.predict_survival_function(X_std)])
train_ibs = integrated_brier_score(y, y, surv_prob, times)

In [95]:
train_ibs

0.20730354536745163

In [98]:
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in model_with_standardization.predict_survival_function(MAASTRO_std)])
test_ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)

In [99]:
test_ibs

0.2229850398004809

In [87]:
model_without_standardization = ComponentwiseGradientBoostingSurvivalAnalysis().fit(X, y)

In [107]:
model_without_standardization.score(X, y)

0.6704473850031506

In [88]:
model_without_standardization.score(MAASTRO_std, y_MAASTRO)

0.6008832459287883

In [102]:
lower, upper = np.percentile(y["time"], [10, 90])    
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in model_without_standardization.predict_survival_function(X)])
train_ibs = integrated_brier_score(y, y, surv_prob, times)

In [103]:
train_ibs

0.20732804667906002

In [104]:
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in model_without_standardization.predict_survival_function(MAASTRO_std)])
test_ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)

In [105]:
test_ibs

0.21451862790761217

#### Train

In [45]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [46]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-29 14:32:34,845] A new study created in memory with name: no-name-96dde80c-105e-46ae-9f07-22e63893577c


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.6446808510638298
Fold 4 C-index: 0.6482889733840305
Fold 5 C-index: 0.6201716738197425
[I 2024-04-29 14:32:35,208] Trial 0 finished with value: 0.5917375988906803 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.5917375988906803.
Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.6446808510638298
Fold 4 C-index: 0.6482889733840305
Fold 5 C-index: 0.6244635193133047
[I 2024-04-29 14:32:37,808] Trial 1 finished with value: 0.5925959679893928 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.5925959679893928.
Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.6065891472868217
Fold 3 C-index: 0.6446808510638298
Fold 

Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.6787234042553192
Fold 4 C-index: 0.623574144486692
Fold 5 C-index: 0.6244635193133047
[I 2024-04-29 14:32:53,948] Trial 19 finished with value: 0.5944615128482231 and parameters: {'subsample': 0.94616158758761, 'dropout_rate': 0.8841892013974624, 'n_estimators': 299, 'learning_rate': 0.04531699534196399}. Best is trial 9 with value: 0.6333466042580751.
Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.5891472868217055
Fold 3 C-index: 0.6446808510638298
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.6201716738197425
[I 2024-04-29 14:32:55,685] Trial 20 finished with value: 0.6302458290642766 and parameters: {'subsample': 0.6004927729383457, 'dropout_rate': 0.7859448591953863, 'n_estimators': 423, 'learning_rate': 0.08455215126922011}. Best is trial 9 with value: 0.6333466042580751.
Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.5891472868217055
Fold 3 C-index: 0.6446808510638298
Fold 4 C-index: 0.6996197718631179
Fold 5

Fold 1 C-index: 0.5956175298804781
Fold 2 C-index: 0.5872093023255814
Fold 3 C-index: 0.6574468085106383
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6223175965665236
[I 2024-04-29 14:33:28,761] Trial 38 finished with value: 0.6255980953653896 and parameters: {'subsample': 0.21215961579513323, 'dropout_rate': 0.10587583963863223, 'n_estimators': 461, 'learning_rate': 0.08947938084969637}. Best is trial 35 with value: 0.6334445170830234.
Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.6046511627906976
Fold 3 C-index: 0.6446808510638298
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.6244635193133047
[I 2024-04-29 14:33:31,396] Trial 39 finished with value: 0.6218543757472259 and parameters: {'subsample': 0.5197301963671256, 'dropout_rate': 0.443442876832168, 'n_estimators': 479, 'learning_rate': 0.05598015324329953}. Best is trial 35 with value: 0.6334445170830234.
Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.6046511627906976
Fold 3 C-index: 0.6446808510638298


...

Fold 3 IBS: 0.2420883508635342
Fold 4 IBS: 0.28861669986805133
Fold 5 IBS: 0.26850778872891123
[I 2024-04-29 14:35:32,844] Trial 19 finished with value: 0.2960586527188663 and parameters: {'subsample': 0.19625604601881638, 'dropout_rate': 0.9930703343219778, 'n_estimators': 132, 'learning_rate': 0.08681602954259124}. Best is trial 7 with value: 0.2291337047072406.
Fold 1 IBS: 0.2449490755369921
Fold 2 IBS: 0.23136100805221904
Fold 3 IBS: 0.21231495466727224
Fold 4 IBS: 0.23604030462642991
Fold 5 IBS: 0.2158097056778787
[I 2024-04-29 14:35:32,998] Trial 20 finished with value: 0.2280950097121584 and parameters: {'subsample': 0.6102628884762309, 'dropout_rate': 0.5558690358940764, 'n_estimators': 52, 'learning_rate': 0.04127909023805986}. Best is trial 20 with value: 0.2280950097121584.
Fold 1 IBS: 0.2456074604293363
Fold 2 IBS: 0.23302758221566516
Fold 3 IBS: 0.21178731577318305
Fold 4 IBS: 0.23616214029004035
Fold 5 IBS: 0.21454946703754782
[I 2024-04-29 14:35:33,160] Trial 21 finished

Fold 1 IBS: 0.2616929141865961
Fold 2 IBS: 0.2584746420928571
Fold 3 IBS: 0.20509225002521414
Fold 4 IBS: 0.24255128592615663
Fold 5 IBS: 0.22016369088716634
[I 2024-04-29 14:35:39,641] Trial 39 finished with value: 0.23759495662359806 and parameters: {'subsample': 0.7913354503723935, 'dropout_rate': 0.49942804580714845, 'n_estimators': 66, 'learning_rate': 0.06892741183938003}. Best is trial 35 with value: 0.22806076229202218.
Fold 1 IBS: 0.3274822159175251
Fold 2 IBS: 0.3801203890074418
Fold 3 IBS: 0.26275327867621484
Fold 4 IBS: 0.302592865334411
Fold 5 IBS: 0.33915386361994393
[I 2024-04-29 14:35:41,875] Trial 40 finished with value: 0.3224205225111073 and parameters: {'subsample': 0.635202567294034, 'dropout_rate': 0.4193126238717278, 'n_estimators': 434, 'learning_rate': 0.053435359182960426}. Best is trial 35 with value: 0.22806076229202218.
Fold 1 IBS: 0.25431839807904555
Fold 2 IBS: 0.24584941441284566
Fold 3 IBS: 0.20889578185906552
Fold 4 IBS: 0.239073073530296
Fold 5 IBS: 0

Fold 2 IBS: 0.29801343453883344
Fold 3 IBS: 0.2043575111283917
Fold 4 IBS: 0.2501181480642662
Fold 5 IBS: 0.23960609341954714
[I 2024-04-29 14:35:46,827] Trial 59 finished with value: 0.2532323254654637 and parameters: {'subsample': 0.8600220350417078, 'dropout_rate': 0.576594985075754, 'n_estimators': 104, 'learning_rate': 0.05792943581299736}. Best is trial 44 with value: 0.22803532502111373.
Fold 1 IBS: 0.24480616451538592
Fold 2 IBS: 0.22872598499241903
Fold 3 IBS: 0.2219822920031368
Fold 4 IBS: 0.23824537864597717
Fold 5 IBS: 0.22249061216421565
[I 2024-04-29 14:35:46,963] Trial 60 finished with value: 0.23125008646422693 and parameters: {'subsample': 0.5182083766842418, 'dropout_rate': 0.4100731778926394, 'n_estimators': 43, 'learning_rate': 0.02007445129499992}. Best is trial 44 with value: 0.22803532502111373.
Fold 1 IBS: 0.24539384312264087
Fold 2 IBS: 0.23153532791807438
Fold 3 IBS: 0.21353666453704093
Fold 4 IBS: 0.23579597885526882
Fold 5 IBS: 0.21608070606317314
[I 2024-04

Fold 4 IBS: 0.24223018529288326
Fold 5 IBS: 0.2066465684530942
[I 2024-04-29 14:35:54,830] Trial 78 finished with value: 0.23265022925229273 and parameters: {'subsample': 0.5484457169084019, 'dropout_rate': 0.22015964511958286, 'n_estimators': 83, 'learning_rate': 0.04941919385503316}. Best is trial 44 with value: 0.22803532502111373.
Fold 1 IBS: 0.28546627650696904
Fold 2 IBS: 0.3380677593689487
Fold 3 IBS: 0.21408985974371417
Fold 4 IBS: 0.26162509079926033
Fold 5 IBS: 0.2504196235788123
[I 2024-04-29 14:35:55,384] Trial 79 finished with value: 0.2699337219995409 and parameters: {'subsample': 0.5921521465840234, 'dropout_rate': 0.8749042812122334, 'n_estimators': 188, 'learning_rate': 0.04027300903668814}. Best is trial 44 with value: 0.22803532502111373.
Fold 1 IBS: 0.24411324640618776
Fold 2 IBS: 0.22881097302349013
Fold 3 IBS: 0.21633320784409502
Fold 4 IBS: 0.23612144672296592
Fold 5 IBS: 0.21873565619800778
[I 2024-04-29 14:35:55,539] Trial 80 finished with value: 0.228822906038

Fold 2 IBS: 0.22944440424173224
Fold 3 IBS: 0.21764182536227877
Fold 4 IBS: 0.23713790468486956
Fold 5 IBS: 0.21744623337572327
[I 2024-04-29 14:35:59,771] Trial 98 finished with value: 0.22917808398847272 and parameters: {'subsample': 0.46010644553945257, 'dropout_rate': 0.40281973549646954, 'n_estimators': 31, 'learning_rate': 0.055235910359091894}. Best is trial 44 with value: 0.22803532502111373.
Fold 1 IBS: 0.2909450482261993
Fold 2 IBS: 0.3516283845294826
Fold 3 IBS: 0.21788174414850223
Fold 4 IBS: 0.2658523928608872
Fold 5 IBS: 0.2652199027303657
[I 2024-04-29 14:36:00,764] Trial 99 finished with value: 0.27830549449908737 and parameters: {'subsample': 0.6151719208120995, 'dropout_rate': 0.631184962372188, 'n_estimators': 278, 'learning_rate': 0.0298041816907381}. Best is trial 44 with value: 0.22803532502111373.


* Best trial for IBS: 
 FrozenTrial(number=44, state=TrialState.COMPLETE, values=[0.22803532502111373], datetime_start=datetime.datetime(2024, 4, 29, 14, 35, 42, 8643

In [48]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.635
train_ibs:  0.228


#### Test

In [49]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [50]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.23864048613297553,
                                              learning_rate=0.014343382745774179,
                                              n_estimators=433,
                                              random_state=123,
                                              subsample=0.40435140119451696)

C-index score: 0.605


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.5020020877800176,
                                              learning_rate=0.04508416432006209,
                                              n_estimators=58, random_state=123,
                                              subsample=0.5795803547524322)

IBS: 0.218


# Component-Wise Gradient boosting without standardization 

In [51]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [52]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-29 14:38:27,715] A new study created in memory with name: no-name-e33cba35-66a9-45d4-b93a-df351c65ce8e


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.6446808510638298
Fold 4 C-index: 0.6482889733840305
Fold 5 C-index: 0.6201716738197425
[I 2024-04-29 14:38:28,050] Trial 0 finished with value: 0.5917375988906803 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.5917375988906803.
Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.6446808510638298
Fold 4 C-index: 0.6482889733840305
Fold 5 C-index: 0.6244635193133047
[I 2024-04-29 14:38:30,598] Trial 1 finished with value: 0.5925959679893928 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.5925959679893928.
Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.6065891472868217
Fold 3 C-index: 0.6446808510638298
Fold 

Fold 1 C-index: 0.5756972111553785
Fold 2 C-index: 0.5891472868217055
Fold 3 C-index: 0.6446808510638298
Fold 4 C-index: 0.6730038022813688
Fold 5 C-index: 0.6158798283261803
[I 2024-04-29 14:38:51,227] Trial 19 finished with value: 0.6196817959296925 and parameters: {'subsample': 0.22885402329391685, 'dropout_rate': 0.9124770129751191, 'n_estimators': 437, 'learning_rate': 0.08677016135502642}. Best is trial 9 with value: 0.6333466042580751.
Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.6046511627906976
Fold 3 C-index: 0.6446808510638298
Fold 4 C-index: 0.6501901140684411
Fold 5 C-index: 0.6244635193133047
[I 2024-04-29 14:38:52,209] Trial 20 finished with value: 0.6243190417978524 and parameters: {'subsample': 0.40774041307597547, 'dropout_rate': 0.6345542498709749, 'n_estimators': 282, 'learning_rate': 0.021614670145549512}. Best is trial 9 with value: 0.6333466042580751.
Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.5891472868217055
Fold 3 C-index: 0.6638297872340425


Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.6046511627906976
Fold 3 C-index: 0.6446808510638298
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.6244635193133047
[I 2024-04-29 14:39:25,829] Trial 38 finished with value: 0.6342049733567876 and parameters: {'subsample': 0.5453560802294278, 'dropout_rate': 0.4972898985304796, 'n_estimators': 497, 'learning_rate': 0.07067813603949202}. Best is trial 38 with value: 0.6342049733567876.
Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.6446808510638298
Fold 4 C-index: 0.6482889733840305
Fold 5 C-index: 0.6201716738197425
[I 2024-04-29 14:39:27,943] Trial 39 finished with value: 0.5917375988906803 and parameters: {'subsample': 0.6942999178388414, 'dropout_rate': 0.5085280680660397, 'n_estimators': 437, 'learning_rate': 0.06892741183938003}. Best is trial 38 with value: 0.6342049733567876.
Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.6046511627906976
Fold 3 C-index: 0.6446808510638298
F

...

Fold 4 IBS: 0.28714816670838966
Fold 5 IBS: 0.26850778872891123
[I 2024-04-29 14:41:21,342] Trial 19 finished with value: 0.29576494608693393 and parameters: {'subsample': 0.19625604601881638, 'dropout_rate': 0.9930703343219778, 'n_estimators': 132, 'learning_rate': 0.08681602954259124}. Best is trial 7 with value: 0.22926527833184956.
Fold 1 IBS: 0.2449490755369921
Fold 2 IBS: 0.23136100805221904
Fold 3 IBS: 0.21231495466727224
Fold 4 IBS: 0.23604030462642991
Fold 5 IBS: 0.2158097056778787
[I 2024-04-29 14:41:21,477] Trial 20 finished with value: 0.2280950097121584 and parameters: {'subsample': 0.6102628884762309, 'dropout_rate': 0.5558690358940764, 'n_estimators': 52, 'learning_rate': 0.04127909023805986}. Best is trial 20 with value: 0.2280950097121584.
Fold 1 IBS: 0.2456074604293363
Fold 2 IBS: 0.23302758221566516
Fold 3 IBS: 0.21178731577318305
Fold 4 IBS: 0.23616214029004035
Fold 5 IBS: 0.21454946703754782
[I 2024-04-29 14:41:21,625] Trial 21 finished with value: 0.22822679314915

Fold 5 IBS: 0.22016369088716634
[I 2024-04-29 14:41:27,627] Trial 39 finished with value: 0.23759495662359806 and parameters: {'subsample': 0.7913354503723935, 'dropout_rate': 0.49942804580714845, 'n_estimators': 66, 'learning_rate': 0.06892741183938003}. Best is trial 35 with value: 0.22806076229202218.
Fold 1 IBS: 0.3274822159175251
Fold 2 IBS: 0.3801203890074418
Fold 3 IBS: 0.26275327867621484
Fold 4 IBS: 0.302592865334411
Fold 5 IBS: 0.33915386361994393
[I 2024-04-29 14:41:29,753] Trial 40 finished with value: 0.3224205225111073 and parameters: {'subsample': 0.635202567294034, 'dropout_rate': 0.4193126238717278, 'n_estimators': 434, 'learning_rate': 0.053435359182960426}. Best is trial 35 with value: 0.22806076229202218.
Fold 1 IBS: 0.25431839807904555
Fold 2 IBS: 0.24584941441284566
Fold 3 IBS: 0.20889578185906552
Fold 4 IBS: 0.239073073530296
Fold 5 IBS: 0.21275686117869264
[I 2024-04-29 14:41:30,032] Trial 41 finished with value: 0.23217870581198907 and parameters: {'subsample':

Fold 1 IBS: 0.2740664401762801
Fold 2 IBS: 0.29801343453883344
Fold 3 IBS: 0.2043575111283917
Fold 4 IBS: 0.2501181480642662
Fold 5 IBS: 0.23960609341954714
[I 2024-04-29 14:41:34,285] Trial 59 finished with value: 0.2532323254654637 and parameters: {'subsample': 0.8600220350417078, 'dropout_rate': 0.576594985075754, 'n_estimators': 104, 'learning_rate': 0.05792943581299736}. Best is trial 44 with value: 0.22803532502111373.
Fold 1 IBS: 0.24480616451538592
Fold 2 IBS: 0.22872598499241903
Fold 3 IBS: 0.2219822920031368
Fold 4 IBS: 0.23824537864597717
Fold 5 IBS: 0.22249061216421565
[I 2024-04-29 14:41:34,412] Trial 60 finished with value: 0.23125008646422693 and parameters: {'subsample': 0.5182083766842418, 'dropout_rate': 0.4100731778926394, 'n_estimators': 43, 'learning_rate': 0.02007445129499992}. Best is trial 44 with value: 0.22803532502111373.
Fold 1 IBS: 0.24539384312264087
Fold 2 IBS: 0.23153532791807438
Fold 3 IBS: 0.21353666453704093
Fold 4 IBS: 0.23579597885526882
Fold 5 IBS:

Fold 3 IBS: 0.2109337973291775
Fold 4 IBS: 0.24223018529288326
Fold 5 IBS: 0.2066465684530942
[I 2024-04-29 14:41:41,942] Trial 78 finished with value: 0.23265022925229273 and parameters: {'subsample': 0.5484457169084019, 'dropout_rate': 0.22015964511958286, 'n_estimators': 83, 'learning_rate': 0.04941919385503316}. Best is trial 44 with value: 0.22803532502111373.
Fold 1 IBS: 0.28546627650696904
Fold 2 IBS: 0.3380677593689487
Fold 3 IBS: 0.21408985974371417
Fold 4 IBS: 0.26162509079926033
Fold 5 IBS: 0.2504196235788123
[I 2024-04-29 14:41:42,443] Trial 79 finished with value: 0.2699337219995409 and parameters: {'subsample': 0.5921521465840234, 'dropout_rate': 0.8749042812122334, 'n_estimators': 188, 'learning_rate': 0.04027300903668814}. Best is trial 44 with value: 0.22803532502111373.
Fold 1 IBS: 0.24411324640618776
Fold 2 IBS: 0.22881097302349013
Fold 3 IBS: 0.21633320784409502
Fold 4 IBS: 0.23612144672296592
Fold 5 IBS: 0.21873565619800778
[I 2024-04-29 14:41:42,579] Trial 80 fini

Fold 5 IBS: 0.2847637022430958
[I 2024-04-29 14:41:46,208] Trial 97 finished with value: 0.29206657343100606 and parameters: {'subsample': 0.5606440507914384, 'dropout_rate': 0.5315723430073316, 'n_estimators': 214, 'learning_rate': 0.04657699840791667}. Best is trial 44 with value: 0.22803532502111373.
Fold 1 IBS: 0.24422005227775972
Fold 2 IBS: 0.22944440424173224
Fold 3 IBS: 0.21764182536227877
Fold 4 IBS: 0.23713790468486956
Fold 5 IBS: 0.21744623337572327
[I 2024-04-29 14:41:46,302] Trial 98 finished with value: 0.22917808398847272 and parameters: {'subsample': 0.46010644553945257, 'dropout_rate': 0.40281973549646954, 'n_estimators': 31, 'learning_rate': 0.055235910359091894}. Best is trial 44 with value: 0.22803532502111373.
Fold 1 IBS: 0.2909450482261993
Fold 2 IBS: 0.3516283845294826
Fold 3 IBS: 0.21788174414850225
Fold 4 IBS: 0.2658523928608872
Fold 5 IBS: 0.2652199027303657
[I 2024-04-29 14:41:47,295] Trial 99 finished with value: 0.27830549449908737 and parameters: {'subsamp

In [53]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.639
train_ibs:  0.228


In [54]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [55]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(dropout_rate=0.1372634977863254,
                                 learning_rate=0.07265284090996345,
                                 n_estimators=274, random_state=123,
                                 subsample=0.6113433675015852)

C-index score: 0.626


GradientBoostingSurvivalAnalysis(dropout_rate=0.5020020877800176,
                                 learning_rate=0.04508416432006209,
                                 n_estimators=58, random_state=123,
                                 subsample=0.5795803547524322)

IBS: 0.227


## Results

In [ ]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

In [ ]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

In [ ]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

In [ ]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

In [ ]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/dfs/minmax/no_selection/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_dfs_minmax_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [ ]:
from datetime import date
today = date.today()
print("Date: ", today)
